<a href="https://colab.research.google.com/github/alyacipta/My-AI-Labs/blob/main/Computer-Vision/Day%202%3A%20Color%20Detection/Color_Detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Util.py**

In [1]:
import numpy as np
import cv2

In [3]:
def get_limits(color):
  # color: BGR format (e.g., [0, 255, 255] for yellow)
  c = np.uint8([[color]])
  hsvC = cv2.cvtColor(c, cv2.COLOR_BGR2HSV)

  hue = hsvC[0][0][0]

  # Handle red hue wrap-around
  if hue >= 165:
    lowerLimit = np.array([hue - 10, 100, 100], dtype=np.uint8)
    upperLimit = np.array([180, 255, 255], dtype=np.uint8)
  elif hue <= 15:
    lowerLimit = np.array([0, 100, 100], dtype=np.uint8)
    upperLimit = np.array([hue + 10, 255, 255], dtype=np.uint8)
  else:
    lowerLimit = np.array([hue - 10, 100, 100], dtype=np.uint8)
    upperLimit = np.array([hue + 10, 255, 255], dtype=np.uint8)

  return lowerLimit, upperLimit

# **Main.py**

In [12]:
import html
import time

# Fungsi JavaScript untuk mengalirkan video secara real-time dan bertukar data dengan Python
def video_stream_realtime():
  js = Javascript('''
    var video;
    var div = null;
    var stream;
    var captureCanvas;
    var imgElement;
    var labelElement;

    var pendingResolve;
    var regrabbing = false;

    function getFrame(resolve) {
      if (!regrabbing) {
        return;
      }
      pendingResolve = resolve;
      captureCanvas.getContext('2d').drawImage(video, 0, 0, 640, 480);
      resolve(captureCanvas.toDataURL('image/jpeg', 0.8));
    }

    async function create() {
      div = document.createElement('div');
      div.style.border = '2px solid black';
      div.style.padding = '10px';
      div.style.width = '660px';
      div.style.margin = '0 auto';

      const label = document.createElement('div');
      label.innerHTML = "<b>Status: Kamera Aktif (Tekan 'Stop' di bawah untuk berhenti)</b>";
      label.style.marginBottom = '10px';
      div.appendChild(label);

      const videoContainer = document.createElement('div');
      videoContainer.style.display = 'flex';
      videoContainer.style.gap = '10px';
      div.appendChild(videoContainer);

      video = document.createElement('video');
      video.style.display = 'block';
      video.width = 320;
      video.height = 240;

      // Elemen untuk menampilkan hasil proses dari Python (Bounding Box/Mask)
      imgElement = document.createElement('img');
      imgElement.width = 320;
      imgElement.height = 240;

      videoContainer.appendChild(video);
      videoContainer.appendChild(imgElement);

      document.body.appendChild(div);

      stream = await navigator.mediaDevices.getUserMedia({video: true});
      video.srcObject = stream;
      await video.play();

      captureCanvas = document.createElement('canvas');
      captureCanvas.width = 640;
      captureCanvas.height = 480;
      regrabbing = true;
    }

    async function streamFrame() {
      if (!regrabbing) return;
      return new Promise(getFrame);
    }

    function showResponse(image_data) {
      imgElement.src = image_data;
    }

    function shutdown() {
      regrabbing = false;
      if (stream) {
        stream.getTracks().forEach(track => track.stop());
      }
      if (div) {
        div.remove();
      }
    }
  ''')
  display(js)

def js_to_image(js_reply):
  image_bytes = b64decode(js_reply.split(',')[1])
  jpg_as_np = np.frombuffer(image_bytes, dtype=np.uint8)
  img = cv2.imdecode(jpg_as_np, flags=1)
  return img

def bbox_to_bytes(bbox_array):
  # Mengonversi array gambar OpenCV kembali ke format base64 untuk ditampilkan di browser
  _, encoded_img = cv2.imencode('.jpg', bbox_array)
  bbox_bytes = b64decode(b64encode(encoded_img).decode('utf-8'))
  return bbox_bytes

In [15]:
from base64 import b64encode

# Jalankan inisialisasi kamera di browser
video_stream_realtime()
eval_js('create()')

print("Memulai deteksi warna kuning secara live. Tekan 'Interrupt Execution' (tombol stop di samping cell) untuk berhenti.")

try:
  while True:
    js_reply = eval_js('streamFrame()')
    if not js_reply:
      break

    frame = js_to_image(js_reply)


    ## Codingan utama proses project "Color Detection"
    # Menentukan target warna kuning (BGR)
    yellow = [0, 255, 255]
    lowerLimit, upperLimit = get_limits(yellow)

    # Proses HSV dan buat Mask
    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
    mask = cv2.inRange(hsv, lowerLimit, upperLimit) # untuk kasih tau ke komputer kalau warna yg kita inginkan, yakni kuning, itu ada di range sekian ke sekian

    # bikin Bounding Box pake teknik Contour dan Draw
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    for contour in contours: # karna warna kuning yg di pilih, jadi saat di countor warna yg dominan itu kuning
      if cv2.contourArea(contour) > 200: # angka 200 itu bisa di adjust sesuai kebutuhan kita, dan ku rasa 200 pas, klo 500 terlalu tinggi akhirnya ada beberapa barang yang g kedetek walau sebenarnya dia kuning
        x, y, w, h = cv2.boundingRect(contour)
        cv2.rectangle(frame, (x, y), (x + w, y + h), (0, 255, 0), 2)
        # cv2.putText(frame, 'Kuning', (x, y - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 0), 2) // opsional, ini hanya untuk kasih keterangan di bawah boundingbox nya


    # Ubah ukuran frame olahan agar muat di panel penampil
    resized_frame = cv2.resize(frame, (320, 240))

    # Kirim kembali frame yang sudah diproses ke browser untuk ditampilkan secara live
    _, encoded_img = cv2.imencode('.jpg', resized_frame)
    img_b64 = b64encode(encoded_img).decode('utf-8')
    img_data = f"data:image/jpeg;base64,{img_b64}"
    eval_js(f"showResponse('{img_data}')")

except KeyboardInterrupt:
  print("Stream dihentikan oleh pengguna.")
finally:
  eval_js('shutdown()')

<IPython.core.display.Javascript object>

Memulai deteksi warna kuning secara live. Tekan 'Interrupt Execution' (tombol stop di samping cell) untuk berhenti.
Stream dihentikan oleh pengguna.
